In [0]:
# Loading my CSV file into a Spark DataFrame for analytics
df = (
    spark.read
         .option("header", "true")      # First row contains column names
         .option("inferSchema", "true") # Automatically detect data types
         .csv("/Volumes/workspace/default/mydatabrick/2019-population_census-report-per-county.csv")
)

# Display the first rows
display(df)

In [0]:
# This line shows the structure of the dataset
df.printSchema()

# This line display the column names
print(df.columns)

In [0]:
from pyspark.sql.functions import col, regexp_replace

# Remove commas from numeric columns and convert them to numeric types
df_clean = (
    df.withColumn(
        "Total_Population19",
        regexp_replace(col("Total_Population19"), ",", "").cast("long")
    )
    .withColumn(
        "Male_population_2019",
        regexp_replace(col("Male populatio 2019"), ",", "").cast("long")
    )
    .withColumn(
        "Female_population_2019",
        regexp_replace(col("Female population 2019"), ",", "").cast("long")
    )
    .withColumn(
        "Households",
        regexp_replace(col("Households"), ",", "").cast("long")
    )
    .withColumn(
        "LandArea",
        regexp_replace(col("LandArea"), ",", "").cast("double")
    )
    .withColumn(
        "Population_Density",
        regexp_replace(col("Population Density"), ",", "").cast("double")
    )
    .withColumn(
        "Population_2009",
        regexp_replace(col("Population in 2009"), ",", "").cast("long")
    )
    .withColumn(
        "Pop_change",
        regexp_replace(regexp_replace(col("Pop_change"), "[()]", ""), ",", "").cast("double")
    )
)

In [0]:
# Verification of the dataset after cleaning
df_clean.printSchema()

In [0]:
#Check for Missing Values in our dataset
from pyspark.sql.functions import col, sum

df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()

In [0]:
#Count the Number of Counties
# Count the total number of counties
county_count = df_clean.count()

print(f"Total Counties: {county_count}")

In [0]:
#Display the Counties
df_clean.select("County").show(47, truncate=False)

In [0]:
#Calculate Kenya's Total Population
from pyspark.sql.functions import sum

df_clean.select(
    sum("Total_Population19").alias("Kenya_Total_Population")
).show()

In [0]:
#Calculate the Average County Population
from pyspark.sql.functions import avg

df_clean.select(
    avg("Total_Population19").alias("Average_County_Population")
).show()

In [0]:
#Find the Most Populated Counties
df_clean.select(
    "County",
    "Total_Population19"
).orderBy(
    "Total_Population19",
    ascending=False
).show(10, truncate=False)

In [0]:
#Find the Least Populated Counties
df_clean.select(
    "County",
    "Total_Population19"
).orderBy(
    "Total_Population19"
).show(10, truncate=False)

In [0]:
#Calculate Average Household Size by County
from pyspark.sql.functions import avg

df_clean.groupBy("County").agg(
    avg("Av_HH_Size").alias("Average_Household_Size")
).orderBy("Average_Household_Size", ascending=False).show(47, truncate=False)

In [0]:
#Calculate Population Density Statistics
df_clean.select(
    "County",
    "Population_Density"
).orderBy(
    "Population_Density",
    ascending=False
).show(10, truncate=False)

In [0]:
#Formulate the MapReduce Job in Spark.\
#Map Phase.----Select the fields we need.
from pyspark.sql.functions import col

mapped_df = df_clean.select(
    col("County"),
    col("Total_Population19"),
    col("Households"),
    col("Av_HH_Size")
)

display(mapped_df)

In [0]:
#Reduce Phase.-------Aggregate by county.
from pyspark.sql.functions import sum, avg

reduced_df = (
    mapped_df
    .groupBy("County")
    .agg(
        sum("Total_Population19").alias("Total_Population"),
        avg("Total_Population19").alias("Average_Population"),
        avg("Av_HH_Size").alias("Average_Household_Size")
    )
)

display(reduced_df)

In [0]:
#Find Interesting Trends from this analyical data
#First, we create a new column.
from pyspark.sql.functions import col

trend_df = df_clean.withColumn(
    "Population_Growth",
    col("Total_Population19") - col("Population_2009")
)

display(trend_df)

In [0]:
#Now we sort counties by growth.
trend_df.select(
    "County",
    "Population_2009",
    "Total_Population19",
    "Population_Growth"
).orderBy(
    "Population_Growth",
    ascending=False
).show(10, truncate=False)

In [0]:
#Counties with Highest Household Size
df_clean.select(
    "County",
    "Av_HH_Size"
).orderBy(
    "Av_HH_Size",
    ascending=False
).show(10, truncate=False)

In [0]:
#Counties with Lowest Household Size
df_clean.select(
    "County",
    "Av_HH_Size"
).orderBy(
    "Av_HH_Size"
).show(10, truncate=False)

In [0]:
#Statistical summary
df_clean.describe().show()